## 1. 라이브러리 및 경로 설정

모델 학습에 필요한 라이브러리를 불러오고,
Training / Validation 이미지와 라벨 데이터 경로를 설정한다.

In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import tensorflow as tf

# --------------------------------------------------
# 프로젝트 기본 경로
# --------------------------------------------------

# 감정인식 프로젝트의 최상위 폴더
PROJECT_ROOT = Path(r"D:\emotion_recognition_project")

# 데이터 폴더
DATA_DIR = PROJECT_ROOT / "02_data"

# --------------------------------------------------
# 컬러 이미지 경로
# --------------------------------------------------

# 224x224 컬러 Training 이미지
COLOR_TRAIN_DIR = DATA_DIR / "processed" / "train"

# 224x224 컬러 Validation 이미지
COLOR_VALID_DIR = DATA_DIR / "processed" / "valid"

# --------------------------------------------------
# 흑백 PNG 이미지 경로
# --------------------------------------------------

# 224x224 흑백 Training 이미지
GRAY_TRAIN_DIR = DATA_DIR / "processed" / "grayscale_png" / "train"

# 224x224 흑백 Validation 이미지
GRAY_VALID_DIR = DATA_DIR / "processed" / "grayscale_png" / "valid"

# --------------------------------------------------
# 라벨 데이터 경로
# --------------------------------------------------

TRAIN_LABEL_DIR = DATA_DIR / "labels" / "train"
VALID_LABEL_DIR = DATA_DIR / "labels" / "valid"

# --------------------------------------------------
# TensorFlow 환경 확인
# --------------------------------------------------

print("TensorFlow 버전:", tf.__version__)
print("GPU 목록:", tf.config.list_physical_devices("GPU"))

# --------------------------------------------------
# 주요 경로 확인
# --------------------------------------------------

print("컬러 Training:", COLOR_TRAIN_DIR)
print("컬러 Validation:", COLOR_VALID_DIR)
print("흑백 Training:", GRAY_TRAIN_DIR)
print("흑백 Validation:", GRAY_VALID_DIR)

TensorFlow 버전: 2.10.1
GPU 목록: []
컬러 Training: D:\emotion_recognition_project\02_data\processed\train
컬러 Validation: D:\emotion_recognition_project\02_data\processed\valid
흑백 Training: D:\emotion_recognition_project\02_data\processed\grayscale_png\train
흑백 Validation: D:\emotion_recognition_project\02_data\processed\grayscale_png\valid


## 2. 이미지와 감정 라벨 연결

전처리된 Training / Validation 이미지의 파일명과
AI-Hub JSON 라벨의 `filename`을 연결하여 학습용 데이터프레임을 생성한다.

In [3]:
# --------------------------------------------------
# 1. 전처리된 컬러 이미지 파일명 수집
# --------------------------------------------------

# Training 폴더 안의 실제 이미지 경로를 가져옴
color_train_files = [
    path
    for path in COLOR_TRAIN_DIR.rglob("*")
    if path.is_file()
]

# Validation 폴더 안의 실제 이미지 경로를 가져옴
color_valid_files = [
    path
    for path in COLOR_VALID_DIR.rglob("*")
    if path.is_file()
]

# 파일명을 set으로 저장
# 예: "abc123.jpg"
train_filenames = {path.name for path in color_train_files}
valid_filenames = {path.name for path in color_valid_files}

print("Training 이미지 수:", len(train_filenames))
print("Validation 이미지 수:", len(valid_filenames))


# --------------------------------------------------
# 2. Training 이미지와 라벨 연결
# --------------------------------------------------

train_records = []

# Training 라벨 JSON 파일을 하나씩 읽음
for json_path in sorted(TRAIN_LABEL_DIR.rglob("*.json")):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # JSON 안의 각 이미지 정보를 확인
    for record in data:

        filename = record["filename"]

        # 실제 사용하는 Training 이미지가 존재할 때만 저장
        if filename in train_filenames:

            train_records.append({
                "filename": filename,

                # AI-Hub 기존 감정 라벨 사용
                "label": record["faceExp_uploader"],

                # 모델이 실제 이미지를 읽을 수 있도록 전체 경로 저장
                "filepath": str(COLOR_TRAIN_DIR / filename)
            })


# --------------------------------------------------
# 3. Validation 이미지와 라벨 연결
# --------------------------------------------------

valid_records = []

for json_path in sorted(VALID_LABEL_DIR.rglob("*.json")):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for record in data:

        filename = record["filename"]

        # 실제 Validation 이미지가 존재할 때만 저장
        if filename in valid_filenames:

            valid_records.append({
                "filename": filename,
                "label": record["faceExp_uploader"],
                "filepath": str(COLOR_VALID_DIR / filename)
            })


# --------------------------------------------------
# 4. DataFrame으로 변환
# --------------------------------------------------

# 리스트 형태의 데이터를 표 형태로 변환
train_df = pd.DataFrame(train_records)
valid_df = pd.DataFrame(valid_records)

print()
print("Training 연결 데이터 수:", len(train_df))
print("Validation 연결 데이터 수:", len(valid_df))

print()
display(train_df.head())

Training 이미지 수: 223578
Validation 이미지 수: 52126

Training 연결 데이터 수: 223578
Validation 연결 데이터 수: 52126



,filename,label,filepath
0,5f656a0f627a3ef96dec882437e3e7ada1c7a877201cf5...,기쁨,D:\emotion_recognition_project\02_data\process...
1,92e7098109470430238876b447f55a978adf237c93b438...,기쁨,D:\emotion_recognition_project\02_data\process...
2,5a30dc193e2c6144184968043c00773270b111a23a3c83...,기쁨,D:\emotion_recognition_project\02_data\process...
3,29b5343b8eca7460aea95cd18db40b015a58ec4db64b71...,기쁨,D:\emotion_recognition_project\02_data\process...
4,4e91bb1317ff7616b99cadf3cafaf8dab49bc6985a0367...,기쁨,D:\emotion_recognition_project\02_data\process...


## 3. 학습용 데이터 일부 구성

학습 시간을 줄이기 위해 전체 Training / Validation 데이터의 약 50%를 사용한다.
각 감정 클래스의 비율이 유지되도록 클래스별로 절반씩 무작위 추출한다.

In [4]:
# --------------------------------------------------
# 1. Training 데이터의 클래스별 50% 추출
# --------------------------------------------------

train_half_df = (
    train_df
    .groupby("label", group_keys=False)
    .sample(frac=0.5, random_state=42)
    .reset_index(drop=True)
)


# --------------------------------------------------
# 2. Validation 데이터의 클래스별 50% 추출
# --------------------------------------------------

valid_half_df = (
    valid_df
    .groupby("label", group_keys=False)
    .sample(frac=0.5, random_state=42)
    .reset_index(drop=True)
)


# --------------------------------------------------
# 3. 추출 결과 확인
# --------------------------------------------------

print("전체 Training 데이터 수:", len(train_df))
print("50% Training 데이터 수:", len(train_half_df))

print()

print("전체 Validation 데이터 수:", len(valid_df))
print("50% Validation 데이터 수:", len(valid_half_df))

print()

print("Training 감정별 데이터 수:")
print(train_half_df["label"].value_counts().sort_index())

print()

print("Validation 감정별 데이터 수:")
print(valid_half_df["label"].value_counts().sort_index())

전체 Training 데이터 수: 223578
50% Training 데이터 수: 111789

전체 Validation 데이터 수: 52126
50% Validation 데이터 수: 26065

Training 감정별 데이터 수:
label
기쁨    15993
당황    16047
분노    16057
불안    16023
상처    15764
슬픔    15839
중립    16066
Name: count, dtype: int64

Validation 감정별 데이터 수:
label
기쁨    3750
당황    3727
분노    3730
불안    3704
상처    3712
슬픔    3740
중립    3702
Name: count, dtype: int64


## 4. MobileNetV2 컬러 이미지 학습

전체 데이터의 50%를 사용하여 MobileNetV2 모델을 학습한다.

컬러 이미지는 224×224×3 크기로 입력하고,
ImageNet 사전학습 가중치를 사용하여 7개 감정을 분류한다.

In [5]:
# --------------------------------------------------
# 1. 학습 기본 설정
# --------------------------------------------------

IMG_SIZE = (224, 224)   # 모델에 입력할 이미지 크기
BATCH_SIZE = 32         # 한 번에 학습할 이미지 수
EPOCHS = 5              # 우선 5회 반복 학습

# 감정 7개를 숫자 라벨로 바꾸기 위한 클래스 순서
class_names = [
    "기쁨",
    "당황",
    "분노",
    "불안",
    "상처",
    "슬픔",
    "중립"
]

# 감정 이름 → 숫자
# 예: 기쁨 → 0, 당황 → 1
label_to_index = {
    label: index
    for index, label in enumerate(class_names)
}


# --------------------------------------------------
# 2. DataFrame의 문자 라벨을 숫자로 변환
# --------------------------------------------------

train_half_df["label_index"] = (
    train_half_df["label"]
    .map(label_to_index)
)

valid_half_df["label_index"] = (
    valid_half_df["label"]
    .map(label_to_index)
)


# --------------------------------------------------
# 3. 이미지 파일을 읽는 함수 정의
# --------------------------------------------------

def load_color_image(filepath, label):

    # 파일 경로를 이용해 이미지 파일 읽기
    image = tf.io.read_file(filepath)

    # JPEG 이미지를 RGB 3채널로 디코딩
    image = tf.io.decode_jpeg(
        image,
        channels=3
    )

    # 이미지 크기를 224x224로 맞춤
    # 이미 전처리되어 있지만 모델 입력 크기를 확실히 맞추기 위해 사용
    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    # MobileNetV2 전처리 방식 적용
    # 픽셀 값을 MobileNetV2가 학습하기 좋은 범위로 변환
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        image
    )

    return image, label


# --------------------------------------------------
# 4. TensorFlow 학습 데이터셋 생성
# --------------------------------------------------

# filepath와 숫자 라벨을 TensorFlow Dataset으로 변환
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_half_df["filepath"].values,
        train_half_df["label_index"].values
    )
)

valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_half_df["filepath"].values,
        valid_half_df["label_index"].values
    )
)


# 각 파일 경로를 실제 이미지 데이터로 변환
train_ds = train_ds.map(
    load_color_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

valid_ds = valid_ds.map(
    load_color_image,
    num_parallel_calls=tf.data.AUTOTUNE
)


# Training 데이터는 순서를 섞음
train_ds = train_ds.shuffle(
    1000,
    seed=42
)


# batch 단위로 묶고 다음 데이터를 미리 준비
train_ds = (
    train_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

valid_ds = (
    valid_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# --------------------------------------------------
# 5. MobileNetV2 모델 생성
# --------------------------------------------------

# ImageNet으로 미리 학습된 MobileNetV2 불러오기
# include_top=False:
# 기존 1000개 ImageNet 분류층은 사용하지 않음
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# 사전학습된 MobileNetV2 부분은 우선 학습하지 않음
base_model.trainable = False


# --------------------------------------------------
# 6. 7개 감정 분류층 추가
# --------------------------------------------------

model = tf.keras.Sequential([
    base_model,

    # 특징맵 전체를 평균내어 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 과적합을 줄이기 위한 Dropout
    tf.keras.layers.Dropout(0.2),

    # 최종 7개 감정 분류
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 7. 모델 학습 설정
# --------------------------------------------------

model.compile(
    # 가중치를 어떻게 업데이트할지 결정
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    # 숫자 형태의 정답 라벨을 사용하므로
    # sparse_categorical_crossentropy 사용
    loss="sparse_categorical_crossentropy",

    # 정확도 확인
    metrics=["accuracy"]
)


# 모델 구조 확인
model.summary()


# --------------------------------------------------
# 8. MobileNetV2 컬러 이미지 학습
# --------------------------------------------------

history_v2_color = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS
)

9406464/9406464 [==============================] - 0s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Funct  (None, 7, 7, 1280)       2257984   
 ional)                                                          
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 7)                 8967      
                                                                 
Total params: 2,266,951
Trainable params: 8,967
Non-trainable params: 2,257,984
_____________________________________________

KeyboardInterrupt: 

### 4-2. MobileNetV2 흑백 이미지 학습

MobileNetV2 컬러 이미지 학습과 동일한 데이터와 학습 조건을 사용하여
흑백 이미지로 1 Epoch 학습을 진행한다.

흑백 이미지는 1채널이지만 ImageNet 사전학습 MobileNetV2의 입력 형식에 맞추기 위해
동일한 흑백 값을 3채널로 변환하여 모델에 입력한다.

컬러와 흑백의 학습 결과를 비교하여 색상 정보가 감정 분류 성능에 미치는 영향을 확인한다.

In [6]:
# --------------------------------------------------
# 1. 흑백 이미지 경로 생성
# --------------------------------------------------

# 컬러 학습에서 사용했던 것과 동일한 이미지 샘플을 사용함
# 파일명은 그대로 유지하고 확장자만 .png로 변경하여
# grayscale_png 폴더의 흑백 이미지와 연결
train_half_df["gray_filepath"] = train_half_df["filename"].apply(
    lambda x: str(GRAY_TRAIN_DIR / f"{Path(x).stem}.png")
)

valid_half_df["gray_filepath"] = valid_half_df["filename"].apply(
    lambda x: str(GRAY_VALID_DIR / f"{Path(x).stem}.png")
)


# --------------------------------------------------
# 2. 흑백 이미지 불러오기 함수
# --------------------------------------------------

def load_gray_image(filepath, label):

    # PNG 이미지 파일 읽기
    image = tf.io.read_file(filepath)

    # 흑백 이미지이므로 1채널로 디코딩
    image = tf.io.decode_png(
        image,
        channels=1
    )

    # MobileNetV2는 3채널 입력을 사용하므로
    # 같은 흑백 값을 R, G, B 3개 채널로 복제
    image = tf.image.grayscale_to_rgb(image)

    # 모델 입력 크기인 224x224로 맞춤
    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    # MobileNetV2의 ImageNet 사전학습 방식에 맞게
    # 픽셀 값을 전처리
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        image
    )

    return image, label


# --------------------------------------------------
# 3. TensorFlow Dataset 생성
# --------------------------------------------------

gray_train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_half_df["gray_filepath"].values,
        train_half_df["label_index"].values
    )
)

gray_valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_half_df["gray_filepath"].values,
        valid_half_df["label_index"].values
    )
)


# 이미지 파일 경로를 실제 이미지 데이터로 변환
gray_train_ds = gray_train_ds.map(
    load_gray_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

gray_valid_ds = gray_valid_ds.map(
    load_gray_image,
    num_parallel_calls=tf.data.AUTOTUNE
)


# Training 데이터 순서를 섞음
# 컬러 실험과 동일하게 seed=42 사용
gray_train_ds = gray_train_ds.shuffle(
    1000,
    seed=42
)


# 이미지를 32장씩 묶고 다음 batch를 미리 준비
gray_train_ds = (
    gray_train_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

gray_valid_ds = (
    gray_valid_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# --------------------------------------------------
# 4. 새로운 MobileNetV2 모델 생성
# --------------------------------------------------

# 컬러 학습에 사용했던 모델을 이어서 학습하면
# 공정한 비교가 아니므로 새로운 MobileNetV2를 다시 생성
base_model_v2_gray = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# 사전학습된 특징 추출 부분은 고정
base_model_v2_gray.trainable = False


# 7개 감정을 분류할 모델 구성
model_v2_gray = tf.keras.Sequential([
    base_model_v2_gray,

    # 특징맵을 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 과적합 완화
    tf.keras.layers.Dropout(0.2),

    # 7개 감정 분류
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 5. 모델 학습 설정
# --------------------------------------------------

model_v2_gray.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# --------------------------------------------------
# 6. MobileNetV2 흑백 이미지 1 Epoch 학습
# --------------------------------------------------

history_v2_gray = model_v2_gray.fit(
    gray_train_ds,
    validation_data=gray_valid_ds,
    epochs=1
)

3494/3494 [==============================] - 1655s 469ms/step - loss: 0.1348 - accuracy: 0.9566 - val_loss: 12.9043 - val_accuracy: 0.1420


### 4-3. MobileNetV2 컬러 / 흑백 1차 비교

전체 데이터의 50%를 사용하여 MobileNetV2 모델을 컬러와 흑백 이미지로 각각 1 Epoch 학습하였다.

| 구분 | Color | Grayscale |
|---|---:|---:|
| Training loss | 0.1364 | 0.1348 |
| Training accuracy | 95.65% | 95.66% |
| Validation loss | 12.9502 | 12.9043 |
| Validation accuracy | 14.20% | 14.20% |
| 학습 시간 | 1342초 | 1655초 |

컬러와 흑백의 Training 정확도와 Validation 정확도는 거의 동일하게 나타났다.
Validation loss는 흑백이 소폭 낮았으나 차이가 매우 작아 성능 우위를 판단하기 어렵다.

학습 시간은 컬러가 약 22분 22초, 흑백이 약 27분 35초로
이번 CPU 환경에서는 컬러 이미지 학습이 더 빠르게 나타났다.

### 4-4. MobileNetV3 Small 컬러 이미지 학습

MobileNetV2와 동일한 데이터와 학습 조건을 사용하여
MobileNetV3 Small 모델을 컬러 이미지로 1 Epoch 학습한다.

동일 조건에서 MobileNetV2와 MobileNetV3 Small의 성능 차이를 비교한다.

In [9]:
# --------------------------------------------------
# 1. MobileNetV3 Small 모델 생성
# --------------------------------------------------

# ImageNet으로 사전학습된 MobileNetV3 Small의 특징 추출 부분 사용
# include_top=False:
# ImageNet용 1000개 분류층은 제외
#
# include_preprocessing=False:
# 앞에서 이미 MobileNetV2 preprocess_input()으로
# 입력값을 -1 ~ 1 범위로 변환했기 때문에
# MobileNetV3 내부 전처리는 사용하지 않음
base_model_v3_color = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=False
)

# 사전학습된 특징 추출 부분의 가중치는 우선 고정
base_model_v3_color.trainable = False


# --------------------------------------------------
# 2. 7개 감정 분류 모델 구성
# --------------------------------------------------

model_v3_color = tf.keras.Sequential([
    base_model_v3_color,

    # 특징맵을 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 과적합을 줄이기 위한 Dropout
    tf.keras.layers.Dropout(0.2),

    # 7개 감정 분류
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 3. 모델 학습 설정
# --------------------------------------------------

model_v3_color.compile(
    # V2와 동일한 Adam optimizer와 learning rate 사용
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    # 숫자 라벨을 사용하므로 sparse categorical loss 사용
    loss="sparse_categorical_crossentropy",

    # 정확도 확인
    metrics=["accuracy"]
)


# 모델 구조 확인
model_v3_color.summary()


# --------------------------------------------------
# 4. MobileNetV3 Small 컬러 이미지 학습
# --------------------------------------------------

# V2 컬러 실험에서 사용한 동일한 train_ds / valid_ds 사용
# 이번에는 1 Epoch만 실행
history_v3_color = model_v3_color.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=1
)

4334752/4334752 [==============================] - 0s 0us/step
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 MobilenetV3small (Functiona  (None, 7, 7, 576)        939120    
 l)                                                              
                                                                 
 global_average_pooling2d_2   (None, 576)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dropout_2 (Dropout)         (None, 576)               0         
                                                                 
 dense_2 (Dense)             (None, 7)                 4039      
                                                                 
Total params: 943,159
Trainable params: 4,039
Non-trainable params: 939,120
_______________________________________________

### 4-5. MobileNetV3 Small 흑백 이미지 학습

MobileNetV3 Small 컬러 이미지 학습과 동일한 데이터와 학습 조건을 사용하여
흑백 이미지로 1 Epoch 학습한다.

흑백 이미지는 3채널로 변환하여 입력하고,
컬러와 흑백 이미지의 성능 및 학습 시간 차이를 비교한다.

In [10]:
# --------------------------------------------------
# 1. MobileNetV3 Small 흑백 모델 생성
# --------------------------------------------------

# ImageNet으로 사전학습된 MobileNetV3 Small의
# 특징 추출 부분을 불러옴
base_model_v3_gray = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),

    # 기존 ImageNet 1000개 분류층은 사용하지 않음
    include_top=False,

    # ImageNet 사전학습 가중치 사용
    weights="imagenet",

    # 흑백 데이터셋에서 이미 -1~1 범위 전처리를 적용했으므로
    # MobileNetV3 내부 전처리는 사용하지 않음
    include_preprocessing=False
)

# 사전학습된 특징 추출 부분의 가중치는 고정
base_model_v3_gray.trainable = False


# --------------------------------------------------
# 2. 7개 감정 분류 모델 구성
# --------------------------------------------------

model_v3_gray = tf.keras.Sequential([
    base_model_v3_gray,

    # 특징맵을 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 과적합 완화를 위한 Dropout
    tf.keras.layers.Dropout(0.2),

    # 7개 감정을 확률로 분류
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 3. 모델 학습 설정
# --------------------------------------------------

model_v3_gray.compile(

    # 다른 실험과 동일한 Adam과 학습률 사용
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    # 숫자로 변환된 감정 라벨을 사용
    loss="sparse_categorical_crossentropy",

    # 정확도를 평가 지표로 사용
    metrics=["accuracy"]
)


# 모델 구조 확인
model_v3_gray.summary()


# --------------------------------------------------
# 4. MobileNetV3 Small 흑백 이미지 학습
# --------------------------------------------------

# 앞에서 MobileNetV2 흑백 학습에 사용했던
# 동일한 gray_train_ds / gray_valid_ds를 사용
history_v3_gray = model_v3_gray.fit(
    gray_train_ds,
    validation_data=gray_valid_ds,

    # 다른 모델들과 동일하게 1 Epoch만 학습
    epochs=1
)

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 MobilenetV3small (Functiona  (None, 7, 7, 576)        939120    
 l)                                                              
                                                                 
 global_average_pooling2d_3   (None, 576)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dropout_3 (Dropout)         (None, 576)               0         
                                                                 
 dense_3 (Dense)             (None, 7)                 4039      
                                                                 
Total params: 943,159
Trainable params: 4,039
Non-trainable params: 939,120
_________________________________________________________________
3494/3494 [==============================] -

### 4-6. MobileNetV3 Small 컬러 / 흑백 1차 비교

전체 데이터의 50%를 사용하여 MobileNetV3 Small 모델을
컬러와 흑백 이미지로 각각 1 Epoch 학습하였다.

| 구분 | Color | Grayscale |
|---|---:|---:|
| Training loss | 0.1385 | 0.1394 |
| Training accuracy | 95.76% | 95.67% |
| Validation loss | 11.2097 | 11.1513 |
| Validation accuracy | 14.20% | 14.20% |
| 학습 시간 | 482초 | 485초 |

컬러와 흑백의 Training 정확도는 거의 비슷하게 나타났고,
Validation 정확도는 두 경우 모두 14.20%로 동일하게 나타났다.

Validation loss는 흑백 이미지에서 조금 더 낮았지만 차이는 매우 작았다.

학습 시간은 컬러 482초, 흑백 485초로 거의 동일하게 나타났다.

따라서 현재 1 Epoch 실험 결과만으로는
MobileNetV3 Small에서 컬러와 흑백 중 어느 쪽이 더 우수하다고 판단하기 어렵다.

### 4-7. EfficientNetB0 컬러 이미지 학습

전처리 단계에서 이미 224×224로 변환한 컬러 이미지를 사용하여
EfficientNetB0 모델을 1 Epoch 학습한다.

EfficientNetB0는 입력 스케일 전처리를 모델 내부에서 처리하므로
별도의 `preprocess_input()`은 적용하지 않는다.

In [11]:
# --------------------------------------------------
# 1. EfficientNetB0용 컬러 이미지 불러오기 함수
# --------------------------------------------------

def load_efficientnet_color(filepath, label):

    # 전처리 단계에서 이미 224x224로 저장한 컬러 이미지 파일을 읽음
    image = tf.io.read_file(filepath)

    # JPEG 파일을 RGB 3채널 이미지로 변환
    # 이미 크기는 224x224이므로 별도의 resize는 하지 않음
    image = tf.io.decode_jpeg(
        image,
        channels=3
    )

    # EfficientNetB0는 입력 스케일 전처리를 모델 내부에서 처리하므로
    # 별도의 preprocess_input()은 적용하지 않음

    return image, label


# --------------------------------------------------
# 2. EfficientNetB0용 컬러 Dataset 생성
# --------------------------------------------------

# 앞에서 만든 전체 데이터의 50% DataFrame 사용
efficient_color_train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_half_df["filepath"].values,
        train_half_df["label_index"].values
    )
)

efficient_color_valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_half_df["filepath"].values,
        valid_half_df["label_index"].values
    )
)


# 파일 경로를 실제 이미지 데이터로 변환
efficient_color_train_ds = efficient_color_train_ds.map(
    load_efficientnet_color,
    num_parallel_calls=tf.data.AUTOTUNE
)

efficient_color_valid_ds = efficient_color_valid_ds.map(
    load_efficientnet_color,
    num_parallel_calls=tf.data.AUTOTUNE
)


# Training 데이터 순서를 섞음
# 기존 실험과 동일하게 random seed는 42로 유지
efficient_color_train_ds = efficient_color_train_ds.shuffle(
    1000,
    seed=42
)


# 기존 실험과 동일한 Batch size 사용
efficient_color_train_ds = (
    efficient_color_train_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

efficient_color_valid_ds = (
    efficient_color_valid_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# --------------------------------------------------
# 3. EfficientNetB0 모델 생성
# --------------------------------------------------

# ImageNet으로 미리 학습된 EfficientNetB0의 특징 추출 부분 사용
base_model_b0_color = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),

    # ImageNet용 기존 분류층은 제외
    include_top=False,

    # ImageNet 사전학습 가중치 사용
    weights="imagenet"
)


# MobileNet 실험과 동일하게
# 사전학습된 특징 추출 부분은 고정
base_model_b0_color.trainable = False


# --------------------------------------------------
# 4. 7개 감정 분류 모델 구성
# --------------------------------------------------

model_b0_color = tf.keras.Sequential([
    base_model_b0_color,

    # 특징맵을 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 기존 실험과 동일한 Dropout 사용
    tf.keras.layers.Dropout(0.2),

    # 7개 감정을 분류하는 최종 출력층
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 5. 모델 학습 설정
# --------------------------------------------------

model_b0_color.compile(

    # 기존 실험과 동일한 Adam optimizer 사용
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    # 숫자형 클래스 라벨을 사용하므로 sparse 방식 사용
    loss="sparse_categorical_crossentropy",

    # 평가 지표는 정확도
    metrics=["accuracy"]
)


# 모델 구조 확인
model_b0_color.summary()


# --------------------------------------------------
# 6. EfficientNetB0 컬러 이미지 1 Epoch 학습
# --------------------------------------------------

history_b0_color = model_b0_color.fit(
    efficient_color_train_ds,
    validation_data=efficient_color_valid_ds,

    # 다른 모델과 동일하게 1 Epoch만 학습
    epochs=1
)

16705208/16705208 [==============================] - 1s 0us/step
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetb0 (Functional)  (None, 7, 7, 1280)       4049571   
                                                                 
 global_average_pooling2d_4   (None, 1280)             0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dropout_4 (Dropout)         (None, 1280)              0         
                                                                 
 dense_4 (Dense)             (None, 7)                 8967      
                                                                 
Total params: 4,058,538
Trainable params: 8,967
Non-trainable params: 4,049,571
_________________________________________________________________
3494/3494 [==============================

### 4-8. EfficientNetB0 흑백 이미지 학습

EfficientNetB0 컬러 이미지 학습과 동일한 데이터와 학습 조건을 사용하여
흑백 이미지로 1 Epoch 학습한다.

전처리 단계에서 생성한 224×224 흑백 PNG 이미지를 사용하며,
EfficientNetB0의 입력 형식에 맞게 흑백 이미지를 3채널로 변환하여 입력한다.

In [12]:
# --------------------------------------------------
# 1. EfficientNetB0용 흑백 이미지 불러오기 함수
# --------------------------------------------------

def load_efficientnet_gray(filepath, label):

    # 전처리 단계에서 저장한 흑백 PNG 이미지 파일 읽기
    image = tf.io.read_file(filepath)

    # PNG 파일을 흑백 1채널 이미지로 변환
    # 이미 224x224이므로 resize는 다시 하지 않음
    image = tf.io.decode_png(
        image,
        channels=1
    )

    # EfficientNetB0는 입력을 3채널로 받으므로
    # 같은 흑백 값을 R, G, B 채널에 각각 복제
    image = tf.image.grayscale_to_rgb(image)

    # EfficientNetB0는 입력 스케일 전처리를 모델 내부에서 처리하므로
    # 별도의 preprocess_input()은 사용하지 않음

    return image, label


# --------------------------------------------------
# 2. EfficientNetB0용 흑백 Dataset 생성
# --------------------------------------------------

efficient_gray_train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_half_df["gray_filepath"].values,
        train_half_df["label_index"].values
    )
)

efficient_gray_valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_half_df["gray_filepath"].values,
        valid_half_df["label_index"].values
    )
)

# 이미지 경로를 실제 흑백 이미지 데이터로 변환
efficient_gray_train_ds = efficient_gray_train_ds.map(
    load_efficientnet_gray,
    num_parallel_calls=tf.data.AUTOTUNE
)

efficient_gray_valid_ds = efficient_gray_valid_ds.map(
    load_efficientnet_gray,
    num_parallel_calls=tf.data.AUTOTUNE
)

# 다른 실험과 동일하게 Training 데이터 순서를 섞음
efficient_gray_train_ds = efficient_gray_train_ds.shuffle(
    1000,
    seed=42
)

# Batch size도 기존 실험과 동일하게 유지
efficient_gray_train_ds = (
    efficient_gray_train_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

efficient_gray_valid_ds = (
    efficient_gray_valid_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# --------------------------------------------------
# 3. EfficientNetB0 흑백 모델 생성
# --------------------------------------------------

base_model_b0_gray = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# ImageNet 사전학습 특징 추출 부분은 고정
base_model_b0_gray.trainable = False


# --------------------------------------------------
# 4. 7개 감정 분류 모델 구성
# --------------------------------------------------

model_b0_gray = tf.keras.Sequential([
    base_model_b0_gray,

    # 특징맵을 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 기존 실험과 동일한 Dropout
    tf.keras.layers.Dropout(0.2),

    # 7개 감정 분류
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 5. 모델 학습 설정
# --------------------------------------------------

model_b0_gray.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_b0_gray.summary()


# --------------------------------------------------
# 6. EfficientNetB0 흑백 이미지 1 Epoch 학습
# --------------------------------------------------

history_b0_gray = model_b0_gray.fit(
    efficient_gray_train_ds,
    validation_data=efficient_gray_valid_ds,
    epochs=1
)

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetb0 (Functional)  (None, 7, 7, 1280)       4049571   
                                                                 
 global_average_pooling2d_5   (None, 1280)             0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dropout_5 (Dropout)         (None, 1280)              0         
                                                                 
 dense_5 (Dense)             (None, 7)                 8967      
                                                                 
Total params: 4,058,538
Trainable params: 8,967
Non-trainable params: 4,049,571
_________________________________________________________________
3494/3494 [==============================] - 1937s 551ms/step - loss: 0.1392 - accuracy: 0.9579 - val_loss

### 4-9. EfficientNetB0 컬러 / 흑백 1차 비교

전체 데이터의 50%를 사용하여 EfficientNetB0 모델을
컬러와 흑백 이미지로 각각 1 Epoch 학습하였다.

| 구분 | Color | Grayscale |
|---|---:|---:|
| Training loss | 0.1422 | 0.1392 |
| Training accuracy | 95.78% | 95.79% |
| Validation loss | 11.0195 | 10.6826 |
| Validation accuracy | 14.20% | 14.20% |
| 학습 시간 | 1921초 | 1937초 |

컬러와 흑백의 Training 정확도는 거의 동일하게 나타났고,
Validation 정확도도 두 경우 모두 14.20%로 동일하였다.

Validation loss는 흑백 이미지에서 조금 더 낮았으며,
학습 시간은 컬러와 흑백이 약 32분으로 거의 비슷하게 나타났다.

따라서 현재 1 Epoch 결과만으로는 컬러와 흑백 중
어느 쪽이 더 우수하다고 판단하기 어렵다.

### 4-10. ResNet18 컬러 이미지 학습

MobileNet 및 EfficientNet 실험과 동일한 50% 데이터와 학습 조건을 사용하여
ImageNet 사전학습 ResNet18 모델을 컬러 이미지로 1 Epoch 학습한다.

전처리 단계에서 이미 생성한 224×224 컬러 이미지를 사용한다.

In [16]:
# ResNet18 모델과 전처리 함수 불러오기
from classification_models.tfkeras import Classifiers

ResNet18, preprocess_input_resnet18 = Classifiers.get("resnet18")

In [15]:
# --------------------------------------------------
# 1. ResNet18용 컬러 이미지 불러오기 함수
# --------------------------------------------------

def load_resnet18_color(filepath, label):

    # 전처리 단계에서 이미 224x224로 저장한 컬러 이미지 파일 읽기
    image = tf.io.read_file(filepath)

    # JPEG 파일을 RGB 3채널 이미지로 변환
    image = tf.io.decode_jpeg(
        image,
        channels=3
    )

    # ResNet18 ImageNet 사전학습 방식에 맞게 입력값 전처리
    image = preprocess_input_resnet18(image)

    return image, label


# --------------------------------------------------
# 2. ResNet18용 컬러 Dataset 생성
# --------------------------------------------------

resnet_color_train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_half_df["filepath"].values,
        train_half_df["label_index"].values
    )
)

resnet_color_valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_half_df["filepath"].values,
        valid_half_df["label_index"].values
    )
)

# 파일 경로를 실제 이미지 데이터로 변환
resnet_color_train_ds = resnet_color_train_ds.map(
    load_resnet18_color,
    num_parallel_calls=tf.data.AUTOTUNE
)

resnet_color_valid_ds = resnet_color_valid_ds.map(
    load_resnet18_color,
    num_parallel_calls=tf.data.AUTOTUNE
)

# 기존 실험과 동일하게 Training 데이터 순서를 섞음
resnet_color_train_ds = resnet_color_train_ds.shuffle(
    1000,
    seed=42
)

# 기존 실험과 동일한 Batch size 사용
resnet_color_train_ds = (
    resnet_color_train_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

resnet_color_valid_ds = (
    resnet_color_valid_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# --------------------------------------------------
# 3. ResNet18 모델 생성
# --------------------------------------------------

# ImageNet으로 사전학습된 ResNet18 특징 추출 부분 사용
base_model_resnet18_color = ResNet18(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# 다른 모델들과 동일하게 backbone은 고정
base_model_resnet18_color.trainable = False


# --------------------------------------------------
# 4. 7개 감정 분류 모델 구성
# --------------------------------------------------

model_resnet18_color = tf.keras.Sequential([
    base_model_resnet18_color,

    # 특징맵을 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 과적합 완화를 위한 Dropout
    tf.keras.layers.Dropout(0.2),

    # 7개 감정 분류
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 5. 모델 학습 설정
# --------------------------------------------------

model_resnet18_color.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_resnet18_color.summary()


# --------------------------------------------------
# 6. ResNet18 컬러 이미지 1 Epoch 학습
# --------------------------------------------------

history_resnet18_color = model_resnet18_color.fit(
    resnet_color_train_ds,
    validation_data=resnet_color_valid_ds,
    epochs=1
)

44920640/44920640 [==============================] - 1s 0us/step
Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 model (Functional)          (None, 7, 7, 512)         11186889  
                                                                 
 global_average_pooling2d_6   (None, 512)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dropout_6 (Dropout)         (None, 512)               0         
                                                                 
 dense_6 (Dense)             (None, 7)                 3591      
                                                                 
Total params: 11,190,480
Trainable params: 3,591
Non-trainable params: 11,186,889
_________________________________________________________________
3494/3494 [============================

### 4-11. ResNet18 흑백 이미지 학습

ResNet18 컬러 이미지 학습과 동일한 데이터와 학습 조건을 사용하여
흑백 이미지로 1 Epoch 학습한다.

전처리 단계에서 생성한 224×224 흑백 PNG 이미지를 사용하며,
ResNet18의 입력 형식에 맞게 흑백 이미지를 3채널로 변환한 뒤
ResNet18 전용 입력 전처리를 적용한다.

In [17]:
# --------------------------------------------------
# 1. ResNet18용 흑백 이미지 불러오기 함수
# --------------------------------------------------

def load_resnet18_gray(filepath, label):

    # 전처리 단계에서 저장한 224x224 흑백 PNG 이미지 읽기
    image = tf.io.read_file(filepath)

    # PNG 파일을 흑백 1채널 이미지로 디코딩
    image = tf.io.decode_png(
        image,
        channels=1
    )

    # ResNet18은 3채널 입력을 사용하므로
    # 동일한 흑백 값을 R, G, B 3개 채널로 복제
    image = tf.image.grayscale_to_rgb(image)

    # ImageNet 사전학습 ResNet18의 입력 방식에 맞게 전처리
    image = preprocess_input_resnet18(image)

    return image, label


# --------------------------------------------------
# 2. ResNet18용 흑백 Dataset 생성
# --------------------------------------------------

resnet_gray_train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_half_df["gray_filepath"].values,
        train_half_df["label_index"].values
    )
)

resnet_gray_valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_half_df["gray_filepath"].values,
        valid_half_df["label_index"].values
    )
)

# 이미지 파일 경로를 실제 이미지 데이터로 변환
resnet_gray_train_ds = resnet_gray_train_ds.map(
    load_resnet18_gray,
    num_parallel_calls=tf.data.AUTOTUNE
)

resnet_gray_valid_ds = resnet_gray_valid_ds.map(
    load_resnet18_gray,
    num_parallel_calls=tf.data.AUTOTUNE
)

# 다른 실험과 동일하게 Training 데이터 순서를 섞음
resnet_gray_train_ds = resnet_gray_train_ds.shuffle(
    1000,
    seed=42
)

# Batch size 32로 묶고 다음 batch를 미리 준비
resnet_gray_train_ds = (
    resnet_gray_train_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

resnet_gray_valid_ds = (
    resnet_gray_valid_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# --------------------------------------------------
# 3. ResNet18 흑백 모델 생성
# --------------------------------------------------

base_model_resnet18_gray = ResNet18(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# 다른 후보 모델과 동일하게 backbone 고정
base_model_resnet18_gray.trainable = False


# --------------------------------------------------
# 4. 7개 감정 분류 모델 구성
# --------------------------------------------------

model_resnet18_gray = tf.keras.Sequential([
    base_model_resnet18_gray,

    # 특징맵을 하나의 벡터로 변환
    tf.keras.layers.GlobalAveragePooling2D(),

    # 다른 실험과 동일한 Dropout
    tf.keras.layers.Dropout(0.2),

    # 7개 감정 분류
    tf.keras.layers.Dense(
        7,
        activation="softmax"
    )
])


# --------------------------------------------------
# 5. 모델 학습 설정
# --------------------------------------------------

model_resnet18_gray.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_resnet18_gray.summary()


# --------------------------------------------------
# 6. ResNet18 흑백 이미지 1 Epoch 학습
# --------------------------------------------------

history_resnet18_gray = model_resnet18_gray.fit(
    resnet_gray_train_ds,
    validation_data=resnet_gray_valid_ds,
    epochs=1
)

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 model_1 (Functional)        (None, 7, 7, 512)         11186889  
                                                                 
 global_average_pooling2d_7   (None, 512)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dropout_7 (Dropout)         (None, 512)               0         
                                                                 
 dense_7 (Dense)             (None, 7)                 3591      
                                                                 
Total params: 11,190,480
Trainable params: 3,591
Non-trainable params: 11,186,889
_________________________________________________________________
3494/3494 [==============================] - 1400s 401ms/step - loss: 0.1408 - accuracy: 0.9560 - val_lo

### 4-12. ResNet18 컬러 / 흑백 1차 비교

전체 데이터의 50%를 사용하여 ResNet18 모델을
컬러와 흑백 이미지로 각각 1 Epoch 학습하였다.

| 구분 | Color | Grayscale |
|---|---:|---:|
| Training loss | 0.1439 | 0.1408 |
| Training accuracy | 95.52% | 95.60% |
| Validation loss | 11.9563 | 12.0331 |
| Validation accuracy | 14.20% | 14.20% |
| 학습 시간 | 1472초 | 1400초 |

컬러와 흑백의 Training 정확도는 거의 비슷하게 나타났고,
Validation 정확도는 두 경우 모두 14.20%로 동일하였다.

Training loss는 흑백이 조금 낮았지만,
Validation loss는 컬러가 조금 더 낮게 나타났다.

학습 시간은 흑백이 컬러보다 약 72초 짧았다.

따라서 현재 1 Epoch 결과만으로는
ResNet18에서 컬러와 흑백 중 어느 쪽이 더 우수하다고 판단하기 어렵다.